In [2]:
import torch
import torch.nn as nn
import os
from torchvision import transforms, datasets
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

class Cecilia(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),               # 224 -> 112
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),               # 112 -> 56
            nn.Flatten(),
            nn.Linear(64 * 56 * 56, 128),  # 注意：输入尺寸 224 时这里才是 64*56*56
            nn.ReLU(),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        return self.model(x)
    
# ---------- 2. 加载测试集（预处理必须与训练时的验证/测试一致）----------
data_root = "./chest_xray_split"          # 你划分的 val 文件夹（或直接用原始 test）
# 注意：训练时验证集用的是 val_test_transform = Normalize(mean=[0.5], std=[0.5])
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])   # 必须与训练时验证集一致
])

# 使用原始的 Chest X-ray test 文件夹（Kaggle 自带的）
test_dataset = ImageFolder("chest_xray/chest_xray/test", transform=val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("测试集类别:", test_dataset.classes)   # 应该是 ['NORMAL', 'PNEUMONIA']
print("测试集样本数:", len(test_dataset))

# ---------- 3. 加载训练好的权重 ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Cecilia().to(device)

weight_path = "checkpoints/best_cecilia.pth"   # 你的 98MB 权重文件路径
if not os.path.exists(weight_path):
    raise FileNotFoundError(f"权重文件不存在: {weight_path}")
model.load_state_dict(torch.load(weight_path, map_location=device))
model.eval()
print("权重加载成功！")

# ---------- 4. 在测试集上评估 ----------
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)          # 注意：ImageFolder 返回的 labels 已经是 long 类型，不需要 squeeze
        outputs = model(imgs)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_acc = correct / total
print(f"\n测试集准确率: {test_acc:.4f} ({correct}/{total})")

# 可选：打印分类报告
from sklearn.metrics import classification_report
print("\n分类报告:")
print(classification_report(all_labels, all_preds, target_names=test_dataset.classes))

测试集类别: ['NORMAL', 'PNEUMONIA']
测试集样本数: 624
权重加载成功！

测试集准确率: 0.8061 (503/624)

分类报告:
              precision    recall  f1-score   support

      NORMAL       0.94      0.52      0.67       234
   PNEUMONIA       0.77      0.98      0.86       390

    accuracy                           0.81       624
   macro avg       0.85      0.75      0.76       624
weighted avg       0.83      0.81      0.79       624



In [3]:
import torch
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import numpy as np

# -------------------------
# 数据集路径
# -------------------------
val_root  = "./chest_xray_split/val"       # 你的验证集
test_root = "./chest_xray/chest_xray/test" # 官方测试集

# -------------------------
# Transform
# -------------------------
val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# -------------------------
# 数据加载器
# -------------------------
batch_size = 32

val_dataset  = ImageFolder(val_root, transform=val_transform)
test_dataset = ImageFolder(test_root, transform=val_transform)

val_loader  = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

num_classes = len(val_dataset.classes)
print("类别:", val_dataset.classes)

# -------------------------
# 模型加载
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

weights_path = "checkpoints/best_resnet18.pth"
model = models.resnet18(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
model.load_state_dict(torch.load(weights_path, map_location=device))
model = model.to(device)
model.eval()

# -------------------------
# 验证/测试函数
# -------------------------
def evaluate(loader, dataset_name="Dataset"):
    all_labels = []
    all_preds  = []
    all_probs  = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(device)
            labels = labels.to(device)

            outputs = model(imgs)
            probs   = torch.softmax(outputs, dim=1)
            preds   = outputs.argmax(dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs[:,1].cpu().numpy())  # PNEUMONIA 概率

    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)

    print(f"\n--- {dataset_name} ---")
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=val_dataset.classes))

    print("Confusion Matrix:")
    print(confusion_matrix(all_labels, all_preds))

    # 计算二分类 AUC
    auc = roc_auc_score(all_labels, all_probs)
    print(f"AUC: {auc:.4f}")

# -------------------------
# 运行验证和测试
# -------------------------
evaluate(val_loader, dataset_name="Validation Set")
evaluate(test_loader, dataset_name="Test Set")

类别: ['NORMAL', 'PNEUMONIA']
device: cuda

--- Validation Set ---
Classification Report:
              precision    recall  f1-score   support

      NORMAL       0.98      0.95      0.96       268
   PNEUMONIA       0.98      0.99      0.99       775

    accuracy                           0.98      1043
   macro avg       0.98      0.97      0.98      1043
weighted avg       0.98      0.98      0.98      1043

Confusion Matrix:
[[254  14]
 [  5 770]]
AUC: 0.9983

--- Test Set ---
Classification Report:
              precision    recall  f1-score   support

      NORMAL       0.99      0.68      0.80       234
   PNEUMONIA       0.84      0.99      0.91       390

    accuracy                           0.88       624
   macro avg       0.91      0.84      0.86       624
weighted avg       0.89      0.88      0.87       624

Confusion Matrix:
[[158  76]
 [  2 388]]
AUC: 0.9843
